In [2]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import os, sys
sys.path.insert(0, '../driver')
from pointing_model import azaltroll_to_theta


In [42]:
csv_filename = './test_validate.csv'
d = pd.read_csv(csv_filename)
d.describe()
d['dev_p_theta1'] = ((d['s_theta1'] - d['p_theta1'] + 180) % 360 - 180)*60
d['dev_p_theta2'] = ((d['s_theta2'] - d['p_theta2'] + 180) % 360 - 180)*60
d['dev_p_theta3'] = ((d['s_theta3'] - d['p_theta3'] + 180) % 360 - 180)*60
d['dev_m_theta1'] = ((d['s_theta1'] - d['m_theta1'] + 180) % 360 - 180)*60
d['dev_m_theta2'] = ((d['s_theta2'] - d['m_theta2'] + 180) % 360 - 180)*60
d['dev_m_theta3'] = ((d['s_theta3'] - d['m_theta3'] + 180) % 360 - 180)*60
d['g_az'] = np.round(d['p_az'] / 5) * 5 
d['g_alt'] = np.round(d['p_alt'] / 5) * 5
d['g_roll'] = np.round(d['p_roll'] / 5) * 5
d.columns

Index(['session_id', 'filename', 'status', 'date_obs', 'site_lat', 'site_lon',
       'p_az', 'p_alt', 'p_roll', 'p_theta1', 'p_theta2', 'p_theta3', 's_az',
       's_alt', 's_roll', 's_theta1', 's_theta2', 's_theta3', 'dev_p_az',
       'dev_p_alt', 'dev_p_roll', 'pixel_scale_arcsec', 'sync_point',
       'alignQ_w', 'alignQ_x', 'alignQ_y', 'alignQ_z', 'm_az', 'm_alt',
       'm_roll', 'm_theta1', 'm_theta2', 'm_theta3', 'dev_m_az', 'dev_m_alt',
       'dev_m_roll', 'dev_p_theta1', 'dev_p_theta2', 'dev_p_theta3',
       'dev_m_theta1', 'dev_m_theta2', 'dev_m_theta3', 'g_az', 'g_alt',
       'g_roll'],
      dtype='object')

In [35]:
d[['m_az', 'm_alt', 'm_roll', 'm_theta1', 'm_theta2', 'm_theta3','dev_m_theta1','dev_m_theta2','dev_m_theta3'  ]].corr(numeric_only=True)

,m_az,m_alt,m_roll,m_theta1,m_theta2,m_theta3,dev_m_theta1,dev_m_theta2,dev_m_theta3
m_az,1.000000,-0.041731,0.054285,0.419471,-0.061233,-0.044114,0.125399,0.030209,-0.240490
m_alt,-0.041731,1.000000,0.035818,-0.010057,0.958077,-0.088032,0.368707,0.173340,-0.239171
m_roll,0.054285,0.035818,1.000000,-0.049225,-0.033017,-0.911466,0.276044,0.369050,-0.059497
m_theta1,0.419471,-0.010057,-0.049225,1.000000,-0.004982,0.039876,0.189627,0.138755,-0.373363
m_theta2,-0.061233,0.958077,-0.033017,-0.004982,1.000000,-0.010559,0.185804,0.075339,-0.088031
m_theta3,-0.044114,-0.088032,-0.911466,0.039876,-0.010559,1.000000,-0.332401,-0.399119,0.050631
dev_m_theta1,0.125399,0.368707,0.276044,0.189627,0.185804,-0.332401,1.000000,0.191814,-0.907284
dev_m_theta2,0.030209,0.173340,0.369050,0.138755,0.075339,-0.399119,0.191814,1.000000,-0.114106
dev_m_theta3,-0.240490,-0.239171,-0.059497,-0.373363,-0.088031,0.050631,-0.907284,-0.114106,1.000000


# Review of Collected Data and Residuals

In [59]:
fig = go.Figure()
legend_offset = 400
d['legend_az'] = d['p_az']/360*100 + legend_offset
d['legend_alt'] = d['p_alt'] + legend_offset
d['legend_roll'] = d['p_roll'] + legend_offset
xfield='date_obs'
for yfield in ['dev_p_az','dev_p_alt','dev_p_roll', 'legend_az', 'legend_alt', 'legend_roll']:
    fig.add_trace(go.Scatter(
        x=d[xfield], y=d[yfield], 
        mode='markers+lines', name=yfield,
        marker=dict(size=6),
    ))

fig.update_layout(
    title=dict(text=f'Axis Residuals vs {xfield}', x=0.5, font=dict(size=24, family='Arial')),
    xaxis_title=f'{xfield}', yaxis_title=f'Axis Residuals (arc-min)', 
    plot_bgcolor='rgba(200, 200, 250, 0.5)',
    hovermode='x unified',
    height=800, width=1400
)

fig.show()

# Theta Space Residuals - BEFORE

In [44]:
fig = px.scatter_matrix(d, dimensions=["dev_p_theta1", "dev_p_theta2", "dev_p_theta3", "p_theta1", "p_theta2", "p_theta3"], color="p_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()

# Theta Space Residuals - AFTER

In [45]:
fig = px.scatter_matrix(d, dimensions=["dev_m_theta1","dev_m_theta2", "dev_m_theta3", "m_theta1", "m_theta2", "m_theta3"], color="m_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals - BEFORE

In [46]:
fig = px.scatter_matrix(d, dimensions=["dev_p_az", "dev_p_alt", "dev_p_roll", "p_theta1", "p_theta2", "p_theta3"], color="p_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()

# Sky Space Residuals - AFTER

In [47]:
fig = px.scatter_matrix(d, dimensions=["dev_m_az", "dev_m_alt", "dev_m_roll", "m_theta1", "m_theta2", "m_theta3"], color="p_roll")
fig.update_layout(height=1000,width=1000 )
fig.show()